# MAT 167 Final Project


In [10]:
# Import dependecies and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import random
import string


### Step 1: Turn hyperlink of nodes into Google Matrix G and check all row sums equal to 1.

In [11]:
import numpy as np

n = 12
nodes = list('ABCDEFGHIJKL')

edges = [
    ('A','I'), ('A','B'), ('A','D'),
    ('B','E'), ('B','D'),
    ('C','B'), ('C','D'),
    # D has no outgoing edges (dangling)
    ('E','G'),
    ('F','B'), ('F','C'), ('F','E'),
    ('G','J'), ('G','E'),
    ('H','F'), ('H','G'), ('H','B'),
    ('I','G'), ('I','J'),
    ('J','K'), ('J','H'),
    ('K','H'),
    ('L','H'), ('L','J'),
]

# Build raw adjacency matrix
idx = {node: i for i, node in enumerate(nodes)}
G = np.zeros((n, n))

for src, dst in edges:
    G[idx[src], idx[dst]] = 1

# Identify dangling nodes (nodes with no outlinks)
dangling_nodes = []
for i, node in enumerate(nodes):
    row_sum = G[i].sum()
    if row_sum == 0:
        dangling_nodes.append(node)
        print(f"Node {node} is dangling (no outlinks) - will set row to 1/{n}")

print(f"\nDangling nodes found: {dangling_nodes}")

# Normalize each row to sum to 1 (row stochastic)
# Fix dangling nodes by setting their row to 1/n everywhere
for i in range(n):
    row_sum = G[i].sum()
    if row_sum == 0:
        G[i] = 1.0/n  # Dangling node: equal probability to all nodes
        print(f"Fixed row {nodes[i]}: set to 1/{n} = {1.0/n:.6f} everywhere")
    else:
        G[i] /= row_sum

df_G = pd.DataFrame(G)
df_G.columns = list(string.ascii_uppercase[:n])
df_G.index = list(string.ascii_uppercase[:n])
display(df_G)

# Check Row Sums
print("\nRow sums:", np.round(G.sum(axis=1), 4))

Node D is dangling (no outlinks) - will set row to 1/12

Dangling nodes found: ['D']
Fixed row D: set to 1/12 = 0.083333 everywhere


,A,B,C,D,E,F,G,H,I,J,K,L
A,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000
B,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
C,0.000000,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333
E,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
F,0.000000,0.333333,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
G,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000
H,0.000000,0.333333,0.000000,0.000000,0.000000,0.333333,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000
I,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000,0.000000
J,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000



Row sums: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Step 2: PageRank vs Eigendecomposition


In [12]:
# Compute eigenvalues and eigenvectors of G^T
eigenvals, eigenvecs = np.linalg.eig(G.T)

# Find the index where eigenvalue is closest to 1
idx_one = np.argmin(np.abs(eigenvals - 1.0))

# Get the corresponding eigenvector
pi_raw = eigenvecs[:, idx_one]

# Normalize to make it a probability distribution (sum to 1)
# Take real part and ensure positive
pi = np.real(pi_raw)
pi = np.abs(pi)
pi = pi / pi.sum()  # Normalize so sum = 1

print(f"Eigenvalue found: {eigenvals[idx_one]:.6f}")
print(f"\nPageRank vector pi (normalized):")
print(np.round(pi, 4))
print(f"\nSum of pi: {pi.sum():.6f}")

# Display as DataFrame with node labels
df_pi = pd.DataFrame({
    'Node': nodes,
    'PageRank': np.round(pi, 6)
})
df_pi = df_pi.sort_values(by="PageRank", ascending=False)
display(df_pi)

Eigenvalue found: 1.000000+0.000000j

PageRank vector pi (normalized):
[0.0049 0.0829 0.0224 0.0593 0.1836 0.0525 0.2394 0.1428 0.0066 0.1304
 0.0701 0.0049]

Sum of pi: 1.000000


,Node,PageRank
6,G,0.239440
4,E,0.183621
7,H,0.142766
9,J,0.130420
1,B,0.082906
10,K,0.070149
3,D,0.059261
5,F,0.052527
2,C,0.022448
8,I,0.006585


### Step 3: Power Method

In [13]:
# Initialize: uniform distribution
pi_iter = np.ones(n) / n
epsilon = 1e-6
max_iter = 1000

print(f"Initial pi_0 (uniform): {np.round(pi_iter, 6)}")
print(f"Tolerance: Epsilon = {epsilon}")
# Iterative power method
for iteration in range(1, max_iter + 1):
    pi_prev = pi_iter.copy()
    pi_iter = pi_iter @ G
    
    # Compute norm of difference
    diff_norm = np.linalg.norm(pi_iter - pi_prev)
    
    if iteration % 10 == 0 or diff_norm <= epsilon:
        print(f"Iteration {iteration}: ||pi_j - pi_j-1|| = {diff_norm:.2e}")
    
    # Check convergence
    if diff_norm <= epsilon:
        print(f"\nConverged after {iteration} iterations!")
        print(f"Final ||pi_j - pi_j-1|| = {diff_norm:.2e}")
        break
else:
    print(f"\nDid not converge after {max_iter} iterations")
    print(f"Final ||pi_j - pi_j-1|| = {diff_norm:.2e}")

# Normalize to ensure sum = 1
pi_iter = pi_iter / pi_iter.sum()

print(f"\nFinal pi (iterative method):")
print(np.round(pi_iter, 6))
print(f"Sum: {pi_iter.sum():.6f}")

# Display as DataFrame
df_pi_iter = pd.DataFrame({
    'Node': nodes,
    'PageRank_Iterative': np.round(pi_iter, 6)
})
df_pi_iter = df_pi_iter.sort_values(by="PageRank_Iterative", ascending=False)
display(df_pi_iter)

Initial pi_0 (uniform): [0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333
 0.083333 0.083333 0.083333 0.083333]
Tolerance: Epsilon = 1e-06
Iteration 10: ||pi_j - pi_j-1|| = 4.06e-03
Iteration 20: ||pi_j - pi_j-1|| = 2.74e-04
Iteration 30: ||pi_j - pi_j-1|| = 1.79e-05
Iteration 40: ||pi_j - pi_j-1|| = 1.18e-06
Iteration 41: ||pi_j - pi_j-1|| = 8.95e-07

Converged after 41 iterations!
Final ||pi_j - pi_j-1|| = 8.95e-07

Final pi (iterative method):
[0.004938 0.082906 0.022448 0.059261 0.183621 0.052527 0.23944  0.142766
 0.006585 0.13042  0.070148 0.004938]
Sum: 1.000000


,Node,PageRank_Iterative
6,G,0.239440
4,E,0.183621
7,H,0.142766
9,J,0.130420
1,B,0.082906
10,K,0.070148
3,D,0.059261
5,F,0.052527
2,C,0.022448
8,I,0.006585


#### Compare Power Method vs Page Rank (Eigendecomposition)

In [14]:
print("=" * 60)
print("COMPARISON: Iterative vs Eigenvalue Decomposition")
print("=" * 60)

# Compare the two PageRank vectors
diff = np.abs(pi_iter - pi)
max_diff = np.max(diff)
mean_diff = np.mean(diff)

print(f"\nDifference between methods:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Mean difference: {mean_diff:.2e}")

# Compare rankings
ranking_eigen = np.argsort(pi)[::-1]
ranking_iter = np.argsort(pi_iter)[::-1]

print(f"\nRankings:")
print(f"  Eigenvalue method: {[nodes[i] for i in ranking_eigen]}")
print(f"  Iterative method:  {[nodes[i] for i in ranking_iter]}")
print(f"  Rankings match? {np.array_equal(ranking_eigen, ranking_iter)}")

print(f"\nBoth methods give the same PageRank vector (within tolerance)")
print(f"Both methods give the same ranking")
print(f"Eigenvalue method and iterative method give the same ranking even though A and L are interchangable rank.")
print(f"This is okay beacause node A and node L has the same eigenvector value.")

COMPARISON: Iterative vs Eigenvalue Decomposition

Difference between methods:
  Max difference: 2.64e-07
  Mean difference: 6.80e-08

Rankings:
  Eigenvalue method: ['G', 'E', 'H', 'J', 'B', 'K', 'D', 'F', 'C', 'I', 'A', 'L']
  Iterative method:  ['G', 'E', 'H', 'J', 'B', 'K', 'D', 'F', 'C', 'I', 'L', 'A']
  Rankings match? False

Both methods give the same PageRank vector (within tolerance)
Both methods give the same ranking
Eigenvalue method and iterative method give the same ranking even though A and L are interchangable rank.
This is okay beacause node A and node L has the same eigenvector value.


### Step 4: Teleportation / Damping Factor Adjustment

In [15]:
# Build adjusted Google matrix with teleportation
# G_tilde = alpha * G + (1 - alpha) * E
# where alpha = 0.85 (damping factor) and E = (1/n) * ones matrix

alpha = 0.85
E = np.ones((n, n)) / n  # 12x12 matrix, each entry = 1/12

G_tilde = alpha * G + (1 - alpha) * E

print(f"Damping factor alpha = {alpha}")
print(f"Teleportation probability = {1 - alpha}")
print(f"\nRow sums of G_tilde: {np.round(G_tilde.sum(axis=1), 6)}")
print(f"All rows sum to 1? {np.allclose(G_tilde.sum(axis=1), 1.0)}")

# --- Eigendecomposition on G_tilde ---
eigenvals_t, eigenvecs_t = np.linalg.eig(G_tilde.T)
idx_one_t = np.argmin(np.abs(eigenvals_t - 1.0))
pi_tilde_raw = eigenvecs_t[:, idx_one_t]
pi_tilde = np.real(pi_tilde_raw)
pi_tilde = np.abs(pi_tilde)
pi_tilde = pi_tilde / pi_tilde.sum()

print(f"\nEigenvalue (should be 1): {np.real(eigenvals_t[idx_one_t]):.6f}")

# --- Power Method on G_tilde ---
pi_tilde_iter = np.ones(n) / n
for iteration in range(1, 1001):
    pi_prev = pi_tilde_iter.copy()
    pi_tilde_iter = pi_tilde_iter @ G_tilde
    diff_norm = np.linalg.norm(pi_tilde_iter - pi_prev)
    if diff_norm <= 1e-6:
        print(f"Power method converged after {iteration} iterations")
        break
pi_tilde_iter = pi_tilde_iter / pi_tilde_iter.sum()

# --- Compare with and without teleportation ---
df_compare = pd.DataFrame({
    'Node': nodes,
    'PageRank (G)':       np.round(pi, 6),
    'PageRank (G_tilde)': np.round(pi_tilde, 6),
    'Power (G_tilde)':    np.round(pi_tilde_iter, 6)
}).sort_values('PageRank (G_tilde)', ascending=False)
display(df_compare)
print("\nRanking with teleportation:", [nodes[i] for i in np.argsort(pi_tilde)[::-1]])


Damping factor alpha = 0.85
Teleportation probability = 0.15000000000000002

Row sums of G_tilde: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
All rows sum to 1? True

Eigenvalue (should be 1): 1.000000
Power method converged after 26 iterations


,Node,PageRank (G),PageRank (G_tilde),Power (G_tilde)
6,G,0.239440,0.200276,0.200277
4,E,0.183621,0.157929,0.157928
7,H,0.142766,0.135403,0.135402
9,J,0.130420,0.120401,0.120401
1,B,0.082906,0.091674,0.091674
3,D,0.059261,0.076328,0.076328
10,K,0.070149,0.069077,0.069077
5,F,0.052527,0.056271,0.056271
2,C,0.022448,0.033850,0.033850
8,I,0.006585,0.022980,0.022980



Ranking with teleportation: ['G', 'E', 'H', 'J', 'B', 'D', 'K', 'F', 'C', 'I', 'L', 'A']


### Step 5: Term-Document Matrix T

In [16]:
# Build Term-Document Matrix T
# T[i, j] = 1 if keyword j appears on page i, else 0

keywords = ['Dodge', 'Nissan', 'Mercedes', 'Audi', 'Jaguar', 'Hyundai',
            'Porsche', 'Lexus', 'Mazda', 'Tesla', 'Cadillac', 'Volkswagen',
            'Alfa Romeo', 'Toyota']

page_keywords = {
    'A': ['Dodge', 'Nissan', 'Mercedes', 'Audi', 'Jaguar', 'Hyundai', 'Porsche', 'Lexus', 'Mazda'],
    'B': ['Dodge', 'Mercedes', 'Jaguar', 'Hyundai', 'Toyota', 'Tesla', 'Cadillac'],
    'C': ['Nissan', 'Mercedes', 'Jaguar', 'Toyota', 'Porsche', 'Mazda', 'Volkswagen'],
    'D': ['Mercedes', 'Hyundai', 'Tesla', 'Porsche', 'Lexus', 'Alfa Romeo'],
    'E': ['Dodge', 'Jaguar', 'Toyota', 'Porsche', 'Cadillac', 'Volkswagen'],
    'F': ['Nissan', 'Mercedes', 'Tesla', 'Porsche', 'Mazda', 'Alfa Romeo', 'Volkswagen'],
    'G': ['Audi', 'Jaguar', 'Tesla', 'Porsche', 'Lexus', 'Mazda', 'Alfa Romeo'],
    'H': ['Audi', 'Hyundai', 'Toyota', 'Cadillac', 'Lexus', 'Alfa Romeo', 'Volkswagen'],
    'I': ['Dodge', 'Nissan', 'Jaguar', 'Hyundai', 'Toyota', 'Cadillac', 'Volkswagen'],
    'J': ['Audi', 'Toyota', 'Porsche', 'Cadillac', 'Lexus', 'Mazda', 'Alfa Romeo'],
    'K': ['Jaguar', 'Hyundai', 'Tesla', 'Porsche', 'Cadillac', 'Mazda', 'Alfa Romeo'],
    'L': ['Nissan', 'Mercedes', 'Audi', 'Jaguar', 'Toyota', 'Tesla', 'Lexus', 'Mazda'],
}

T = np.zeros((n, len(keywords)), dtype=int)
for i, node in enumerate(nodes):
    for j, kw in enumerate(keywords):
        if kw in page_keywords[node]:
            T[i, j] = 1

df_T = pd.DataFrame(T, index=nodes, columns=keywords)
print(f"Term-Document Matrix T  ({n} pages × {len(keywords)} keywords)")
display(df_T)
print(f"\nKeywords per page (row sums):")
print(pd.Series(T.sum(axis=1), index=nodes).to_string())
print(f"\nPages per keyword (col sums):")
print(pd.Series(T.sum(axis=0), index=keywords).to_string())


Term-Document Matrix T  (12 pages × 14 keywords)


,Dodge,Nissan,Mercedes,Audi,Jaguar,Hyundai,Porsche,Lexus,Mazda,Tesla,Cadillac,Volkswagen,Alfa Romeo,Toyota
A,1,1,1,1,1,1,1,1,1,0,0,0,0,0
B,1,0,1,0,1,1,0,0,0,1,1,0,0,1
C,0,1,1,0,1,0,1,0,1,0,0,1,0,1
D,0,0,1,0,0,1,1,1,0,1,0,0,1,0
E,1,0,0,0,1,0,1,0,0,0,1,1,0,1
F,0,1,1,0,0,0,1,0,1,1,0,1,1,0
G,0,0,0,1,1,0,1,1,1,1,0,0,1,0
H,0,0,0,1,0,1,0,1,0,0,1,1,1,1
I,1,1,0,0,1,1,0,0,0,0,1,1,0,1
J,0,0,0,1,0,0,1,1,1,0,1,0,1,1



Keywords per page (row sums):
A    9
B    7
C    7
D    6
E    6
F    7
G    7
H    7
I    7
J    7
K    7
L    8

Pages per keyword (col sums):
Dodge         4
Nissan        5
Mercedes      6
Audi          5
Jaguar        8
Hyundai       6
Porsche       8
Lexus         6
Mazda         7
Tesla         6
Cadillac      6
Volkswagen    5
Alfa Romeo    6
Toyota        7


### Step 6: Search Query — Rank Relevant Pages by PageRank

In [ ]:
def search(query_terms, T, pi_tilde, nodes, keywords, mode='OR', exclude_terms=None):
    """
    Find and rank relevant pages by PageRank given a user query.

    Steps (per FinalProjectDetails.pdf):
      13. Build binary query vector q
      14. Compute d = T @ q  (equivalent to d^T = q^T T)
          d[i] counts how many query keywords appear on page i
      15. Filter pages by mode, then rank by PageRank score (pi_tilde)

    mode: 'OR'  - pages containing AT LEAST ONE query term (SINGLE or OR search)
          'AND' - pages containing ALL query terms
          'NOT' - pages containing query_terms but NOT exclude_terms
    """
    # Step 13: Build binary query vector q
    q = np.array([1 if kw in query_terms else 0 for kw in keywords])

    # Step 14: Compute d = T @ q
    d = T @ q

    # Determine which pages are responsive based on search mode
    if mode == 'NOT' and exclude_terms:
        q_excl = np.array([1 if kw in exclude_terms else 0 for kw in keywords])
        d_excl = T @ q_excl
        relevant_mask = (d > 0) & (d_excl == 0)
    elif mode == 'AND':
        relevant_mask = d == len(query_terms)
    else:
        relevant_mask = d > 0

    relevant_nodes    = [nodes[i] for i in range(n) if relevant_mask[i]]
    relevant_pagerank = pi_tilde[relevant_mask]

    if len(relevant_nodes) == 0:
        print('No pages found for this query.')
        return None

    # Step 15: Rank responsive pages by PageRank (descending)
    order         = np.argsort(relevant_pagerank)[::-1]
    ranked_nodes  = [relevant_nodes[i] for i in order]
    ranked_scores = relevant_pagerank[order]

    results = pd.DataFrame({
        'Rank'            : range(1, len(ranked_nodes) + 1),
        'Page'            : ranked_nodes,
        'PageRank Score'  : np.round(ranked_scores, 6),
        'Keywords Matched': [d[nodes.index(p)] for p in ranked_nodes]
    })
    return results


#### Search Method

For each query we follow steps 13–15 from the project specification:

1. **Build query vector q** — binary vector of length 14. q[j] = 1 if keyword j is in the query, else 0.
2. **Compute d = T @ q** — relevance score vector. d[i] counts how many query keywords appear on page i.
3. **Filter by search mode:**
   - *SINGLE / OR*: keep pages where d[i] > 0
   - *AND*: keep pages where d[i] == number of query terms
   - *BUT NOT*: keep pages where d[i] > 0 AND excluded term is absent
4. **Rank filtered pages** by PageRank score from G̃, highest first.


#### Query 1: SINGLE Search — "Dodge"

A user searches for **Dodge**. Which pages contain this keyword, and in what order should they be returned?

In [ ]:
q1 = np.array([1 if kw == 'Dodge' else 0 for kw in keywords])
print(f'Keywords : {keywords}')
print(f'Query q  : {q1}')
print()
d1 = T @ q1
print('Relevance scores d = T @ q:')
print(pd.Series(d1, index=nodes).to_string())
print()
r1 = search(['Dodge'], T, pi_tilde, nodes, keywords, mode='OR')
print('Search Results — SINGLE Search: Dodge')
display(r1)


#### Query 2: AND Search — "Hyundai AND Porsche"

A user searches for **Hyundai AND Porsche**. Only pages containing **both** keywords are responsive.

In [ ]:
q2 = np.array([1 if kw in ['Hyundai', 'Porsche'] else 0 for kw in keywords])
print(f'Keywords : {keywords}')
print(f'Query q  : {q2}')
print()
d2 = T @ q2
print('Relevance scores d = T @ q (2 = both keywords present):')
print(pd.Series(d2, index=nodes).to_string())
print()
r2 = search(['Hyundai', 'Porsche'], T, pi_tilde, nodes, keywords, mode='AND')
print('Search Results — AND Search: Hyundai AND Porsche')
display(r2)


#### Query 3: OR Search — "Jaguar OR Toyota"

A user searches for **Jaguar OR Toyota**. Pages containing **at least one** of these keywords are responsive.

In [ ]:
q3 = np.array([1 if kw in ['Jaguar', 'Toyota'] else 0 for kw in keywords])
print(f'Keywords : {keywords}')
print(f'Query q  : {q3}')
print()
d3 = T @ q3
print('Relevance scores d = T @ q (>0 means at least one keyword matches):')
print(pd.Series(d3, index=nodes).to_string())
print()
r3 = search(['Jaguar', 'Toyota'], T, pi_tilde, nodes, keywords, mode='OR')
print('Search Results — OR Search: Jaguar OR Toyota')
display(r3)


#### Query 4: BUT NOT Search — "Alfa Romeo BUT NOT Volkswagen"

A user searches for **Alfa Romeo BUT NOT Volkswagen**. Pages must contain **Alfa Romeo** and must **not** contain **Volkswagen**.

In [ ]:
q4      = np.array([1 if kw == 'Alfa Romeo' else 0 for kw in keywords])
q4_excl = np.array([1 if kw == 'Volkswagen' else 0 for kw in keywords])
print(f'Keywords    : {keywords}')
print(f'Query q     : {q4}')
print(f'Exclusion q : {q4_excl}')
print()
d4      = T @ q4
d4_excl = T @ q4_excl
print('Relevance scores (Alfa Romeo):')
print(pd.Series(d4, index=nodes).to_string())
print()
print('Exclusion scores (Volkswagen — pages > 0 are removed):')
print(pd.Series(d4_excl, index=nodes).to_string())
print()
r4 = search(['Alfa Romeo'], T, pi_tilde, nodes, keywords, mode='NOT', exclude_terms=['Volkswagen'])
print('Search Results — BUT NOT Search: Alfa Romeo BUT NOT Volkswagen')
display(r4)
